# Acne 이진 분류 학습 노트북

이 노트북은 EfficientNet-B0 기반으로 Non-Acne(0) vs Acne(1) 이진 분류 모델을 학습합니다.
- 데이터: `skinseal-model/train` (ImageFolder 구조)
- 저장: `skinseal-pythonAI/models/best_acne_model.pth`

환경: PyTorch, Torchvision이 설치되어 있어야 합니다. (requirements.txt 참고)

In [12]:
# 기본 설정 및 버전 확인
import os, sys, math, time
from pathlib import Path
import torch, torchvision
print(f'Torch: {torch.__version__}, Torchvision: {torchvision.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Torch: 2.7.1+cu118, Torchvision: 0.22.1+cu118
Device: cuda


In [ ]:
# 경로/하이퍼파라미터 설정
# 조정 가능한 하이퍼파라미터에는 ★ 표시가 붙어있습니다.
from dataclasses import dataclass
from typing import Tuple

PROJECT_ROOT = Path.cwd()  # 이 노트북이 위치한 skinseal-model 폴더
TRAIN_DIR = PROJECT_ROOT / 'train'  # ImageFolder
OUT_PATH = PROJECT_ROOT.parent / 'skinseal-pythonAI' / 'models' / 'best_acne_model.pth'

# ★ 조정 가능: 입력 이미지 크기 (모델에 따라 변경)
IMG_SIZE = (224, 224)  # ★
# ★ 조정 가능: 배치 크기 (메모리/성능 균형)
BATCH_SIZE = 32  # ★
# ★ 조정 가능: 에폭 수 (학습 길이)
EPOCHS = 5  # 데모용, 필요시 늘리세요  ★
# ★ 조정 가능: 검증 비율
VAL_RATIO = 0.15  # ★
# ★ 조정 가능: 학습률
LR = 1e-3  # ★
# ★ 조정 가능: 가중치 감쇠 (정규화)
WEIGHT_DECAY = 1e-4  # ★
# ★ 조정 가능(테스트 환경에 따라): DataLoader 워커 수
NUM_WORKERS = 4  # ★ (Windows의 경우 0 권장)
# ★ 조정 가능: 사전학습 사용 여부
USE_PRETRAINED = False  # 네트워크 제약 환경을 고려해 기본 False  ★

print('TRAIN_DIR:', TRAIN_DIR)
print('OUT_PATH:', OUT_PATH)

TRAIN_DIR: d:\2ckvmfhwpr\skinseal-model\train
OUT_PATH: d:\2ckvmfhwpr\skinseal-pythonAI\models\best_acne_model.pth


In [14]:
# 진단 셀 1: 환경/디바이스/버전 확인
import torch, torchvision, sys, platform, os
from pathlib import Path
print('Python:', sys.executable)
print('Python version:', sys.version.splitlines()[0])
print('Platform:', platform.platform())
print(f'Torch: {torch.__version__}, Torchvision: {torchvision.__version__}')
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(' Device', i, torch.cuda.get_device_name(i))
    # 현재 GPU 메모리 상태
    print('cuda memory allocated:', torch.cuda.memory_allocated() / (1024**2), 'MB')
    print('cuda memory reserved:', torch.cuda.memory_reserved() / (1024**2), 'MB')
print('Current working dir:', Path.cwd())
print('Notebook kernel PID (if available):', os.getpid())

Python: c:\Users\USER\anaconda3\python.exe
Python version: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
Platform: Windows-11-10.0.26100-SP0
Torch: 2.7.1+cu118, Torchvision: 0.22.1+cu118
CUDA available: True
CUDA device count: 1
 Device 0 NVIDIA GeForce RTX 4070 Ti SUPER
cuda memory allocated: 96.31591796875 MB
cuda memory reserved: 3214.0 MB
Current working dir: d:\2ckvmfhwpr\skinseal-model
Notebook kernel PID (if available): 25908


In [15]:
# 진단 셀 2: 데이터셋 크기 및 클래스 분포 확인
from torchvision import datasets
from collections import Counter
TRAIN_DIR = Path.cwd() / 'train'
print('TRAIN_DIR exists:', TRAIN_DIR.exists())
if TRAIN_DIR.exists():
    ds = datasets.ImageFolder(str(TRAIN_DIR))
    print('전체 샘플 수:', len(ds))
    counts = Counter([p for p in ds.targets])
    idx_to_class = {v:k for k,v in ds.class_to_idx.items()}
    for idx, cnt in counts.items():
        print(f"  class {idx} ({idx_to_class.get(idx,'?')}): {cnt}")
    print('샘플 파일들:')
    for i, (p, _) in enumerate(ds.samples[:5]):
        print(' ', i+1, p)
else:
    print('Train directory가 없어 경로를 확인하세요.')

TRAIN_DIR exists: True
전체 샘플 수: 13898
  class 0 (Acne): 593
  class 1 (Actinic_Keratosis): 748
  class 2 (Benign_tumors): 1093
  class 3 (Bullous): 504
  class 4 (Candidiasis): 248
  class 5 (DrugEruption): 547
  class 6 (Eczema): 1010
  class 7 (Infestations_Bites): 524
  class 8 (Lichen): 553
  class 9 (Lupus): 311
  class 10 (Moles): 361
  class 11 (Psoriasis): 820
  class 12 (Rosacea): 254
  class 13 (Seborrh_Keratoses): 455
  class 14 (SkinCancer): 693
  class 15 (Sun_Sunlight_Damage): 312
  class 16 (Tinea): 923
  class 17 (Unknown_Normal): 1651
  class 18 (Vascular_Tumors): 543
  class 19 (Vasculitis): 461
  class 20 (Vitiligo): 714
  class 21 (Warts): 580
샘플 파일들:
  1 d:\2ckvmfhwpr\skinseal-model\train\Acne\07Acne081101.jpeg
  2 d:\2ckvmfhwpr\skinseal-model\train\Acne\07AcnePittedScars.jpeg
  3 d:\2ckvmfhwpr\skinseal-model\train\Acne\07AcnePittedScars1.jpeg
  4 d:\2ckvmfhwpr\skinseal-model\train\Acne\141__ProtectWyJQcm90ZWN0Il0_FocusFillWzI5NCwyMjIsIngiLDFd.jpeg
  5 d:\2ckvmfhwp

In [16]:
# 진단 셀 3: DataLoader 배치/로드 속도 테스트 (최대 10 배치 측정)
import time
from torch.utils.data import DataLoader
from torchvision import transforms
IMG_SIZE = (224,224)
tf = transforms.Compose([transforms.Resize(IMG_SIZE), transforms.ToTensor()])
try:
    ds = datasets.ImageFolder(str(TRAIN_DIR), transform=tf)
    # 노트: 원래 노트북의 NUM_WORKERS, BATCH_SIZE를 사용하려면 변수 값을 맞춰주세요
    BATCH_SIZE = 32
    NUM_WORKERS = 0  # Windows 환경에서는 우선 0으로 테스트 권장
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    it = iter(loader)
    n = min(10, len(loader))
    t0 = time.time()
    for i in range(n):
        imgs, labels = next(it)
        if torch.cuda.is_available():
            imgs = imgs.cuda(non_blocking=True)
    t1 = time.time()
    print(f'로딩/준비된 {n} 배치 처리 시간: {t1-t0:.2f}초 (배치 크기={BATCH_SIZE})')
    print(f'평균 배치 처리 시간: {(t1-t0)/n:.3f}초')
except Exception as e:
    print('DataLoader 테스트 중 오류:', e)

로딩/준비된 10 배치 처리 시간: 2.09초 (배치 크기=32)
평균 배치 처리 시간: 0.209초


In [17]:
# 진단 셀 4: (CUDA 사용 시) 간단한 GPU 메모리 점검 및 nvidia-smi 시도
import subprocess
if torch.cuda.is_available():
    torch.cuda.synchronize()
    print('메모리 할당(allocated):', torch.cuda.memory_allocated() / (1024**2), 'MB')
    print('메모리 예약(reserved):', torch.cuda.memory_reserved() / (1024**2), 'MB')
    try:
        cmd = 'nvidia-smi'
        print('--- nvidia-smi ---')
        out = subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, universal_newlines=True)
        print(out)
    except Exception as e:
        print('nvidia-smi 실행 불가 또는 오류:', e)
else:
    print('CUDA 사용 불가: GPU 점검 건너뜀')

메모리 할당(allocated): 112.96826171875 MB
메모리 예약(reserved): 3214.0 MB
--- nvidia-smi ---
Wed Oct 29 10:24:43 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.57                 Driver Version: 581.57         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4070 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   34C    P8              9W /  285W |    4216MiB /  16376MiB |     40%      Default |
|                                         |            

In [18]:
# 데이터셋/로더 구성 (Acne=1, 그 외=0)
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset, Subset

assert TRAIN_DIR.exists(), f'Train dir not found: {TRAIN_DIR}'

train_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

base_train = datasets.ImageFolder(str(TRAIN_DIR), transform=train_tf)
base_val = datasets.ImageFolder(str(TRAIN_DIR), transform=val_tf)

class_to_idx = base_train.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}
print('Detected classes:', class_to_idx)

class BinaryWrapper(Dataset):
    def __init__(self, base):
        self.base = base
        self.targets = [1 if idx_to_class[y]=='Acne' else 0 for y in base.targets]
    def __len__(self):
        return len(self.base)
    def __getitem__(self, i):
        x, _ = self.base[i]
        return x, self.targets[i]

bin_train = BinaryWrapper(base_train)
bin_val = BinaryWrapper(base_val)
len(bin_train), len(bin_val)

Detected classes: {'Acne': 0, 'Actinic_Keratosis': 1, 'Benign_tumors': 2, 'Bullous': 3, 'Candidiasis': 4, 'DrugEruption': 5, 'Eczema': 6, 'Infestations_Bites': 7, 'Lichen': 8, 'Lupus': 9, 'Moles': 10, 'Psoriasis': 11, 'Rosacea': 12, 'Seborrh_Keratoses': 13, 'SkinCancer': 14, 'Sun_Sunlight_Damage': 15, 'Tinea': 16, 'Unknown_Normal': 17, 'Vascular_Tumors': 18, 'Vasculitis': 19, 'Vitiligo': 20, 'Warts': 21}


(13898, 13898)

In [19]:
# Stratified split 구현
import random
def split_stratified_indices(labels, val_ratio: float, seed: int = 42):
    from collections import defaultdict
    n = len(labels)
    by_class = defaultdict(list)
    for i, y in enumerate(labels):
        by_class[y].append(i)
    val_idx, train_idx = [], []
    rnd = random.Random(seed)
    for arr in by_class.values():
        rnd.shuffle(arr)
        k = max(1, int(len(arr)*val_ratio))
        val_idx.extend(arr[:k])
        train_idx.extend(arr[k:])
    return train_idx, val_idx

train_idx, val_idx = split_stratified_indices(bin_train.targets, VAL_RATIO, 42)
train_ds = Subset(bin_train, train_idx)
val_ds = Subset(bin_val, val_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
len(train_ds), len(val_ds)

(11815, 2083)

In [20]:
# 모델 구성 (EfficientNet-B0)
import torch.nn as nn
from torchvision import models

def build_model(num_classes: int = 2, use_pretrained: bool = False):
    try:
        if use_pretrained:
            m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        else:
            m = models.efficientnet_b0(weights=None)
    except Exception:
        m = models.efficientnet_b0(weights=None)
    in_features = m.classifier[1].in_features
    m.classifier = nn.Sequential(nn.Dropout(p=0.2, inplace=True), nn.Linear(in_features, num_classes))
    return m

model = build_model(num_classes=2, use_pretrained=USE_PRETRAINED).to(device)
sum(p.numel() for p in model.parameters())

4010110

In [21]:
# 학습 루프 (결과 저장 강화: 모델, 메트릭, 로그를 타임스탬프 폴더에 저장)
import torch.optim as optim
import torch
from tqdm import tqdm
import time
from datetime import datetime
import json
import shutil

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def accuracy_and_f1(outputs, targets):
    with torch.no_grad():
        preds = outputs.argmax(dim=1)
        correct = (preds == targets).sum().item()
        acc = correct / targets.numel()
        tp = int(((preds==1) & (targets==1)).sum().item())
        fp = int(((preds==1) & (targets==0)).sum().item())
        fn = int(((preds==0) & (targets==1)).sum().item())
        precision = tp/(tp+fp) if (tp+fp)>0 else 0.0
        recall = tp/(tp+fn) if (tp+fn)>0 else 0.0
        f1 = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0.0
    return acc, f1

best_f1 = -1.0
best_state = None
LOG_INTERVAL = 50  # 배치 단위 출력 주기

# 기록용 저장소
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
log_lines = []

try:
    for epoch in range(1, EPOCHS+1):
        epoch_t0 = time.time()
        model.train()
        run_loss = 0.0
        # tqdm으로 에포크 내부 배치 진행률 표시
        pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch}/{EPOCHS}", ncols=100)
        for batch_idx, (imgs, ys) in pbar:
            imgs = imgs.to(device)
            ys = torch.as_tensor(ys, dtype=torch.long, device=device)
            optimizer.zero_grad(set_to_none=True)
            outs = model(imgs)
            loss = criterion(outs, ys)
            loss.backward()
            optimizer.step()

            run_loss += float(loss.item()) * imgs.size(0)

            # 주기적으로 상태 업데이트 (tqdm 텍스트와 표준 출력)
            if (batch_idx + 1) % LOG_INTERVAL == 0 or (batch_idx + 1) == len(train_loader):
                lr = optimizer.param_groups[0]['lr'] if optimizer.param_groups else float('nan')
                msg = f"epoch={epoch} batch={batch_idx+1}/{len(train_loader)} loss={loss.item():.4f} lr={lr:.6f}"
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                    msg += f" mem_alloc={torch.cuda.memory_allocated()/(1024**2):.1f}MB"
                pbar.set_postfix_str(msg)
                log_lines.append(msg)

        train_loss = run_loss / max(1, len(train_loader.dataset))

        # Validation
        model.eval()
        val_loss, total_acc, total_f1, cnt = 0.0, 0.0, 0.0, 0
        with torch.no_grad():
            for imgs, ys in tqdm(val_loader, desc=f"Val {epoch}", ncols=100, leave=False):
                imgs = imgs.to(device)
                ys = torch.as_tensor(ys, dtype=torch.long, device=device)
                outs = model(imgs)
                loss = criterion(outs, ys)
                val_loss += float(loss.item()) * imgs.size(0)
                acc, f1 = accuracy_and_f1(outs, ys)
                total_acc += acc * imgs.size(0)
                total_f1 += f1 * imgs.size(0)
                cnt += imgs.size(0)

        val_loss = val_loss / max(1, len(val_loader.dataset))
        val_acc = total_acc / max(1, cnt)
        val_f1 = total_f1 / max(1, cnt)
        epoch_time = time.time() - epoch_t0
        summary = f"Epoch [{epoch}/{EPOCHS}] train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f} time={epoch_time:.1f}s"
        print(summary)
        log_lines.append(summary)

        # 기록 저장
        history['train_loss'].append(float(train_loss))
        history['val_loss'].append(float(val_loss))
        history['val_acc'].append(float(val_acc))
        history['val_f1'].append(float(val_f1))

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {
                'epoch': epoch,
                'val_f1': float(best_f1),
                'model_state_dict': model.state_dict(),
            }
except KeyboardInterrupt:
    print('Interrupted by user — saving current best state (if any) and exiting...')

# 저장: 타임스탬프 폴더 생성 후 결과 저장(모델, 메트릭, 메타데이터, 로그)
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
if best_state is None:
    best_state = {'epoch': 0, 'val_f1': 0.0, 'model_state_dict': model.state_dict()}

# 모델명(폴더명에 사용): OUT_PATH 파일명에서 'best_' 제거
model_name = OUT_PATH.stem.replace('best_', '')
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
out_dir = OUT_PATH.parent / f"{model_name}_{timestamp}"
out_dir.mkdir(parents=True, exist_ok=True)

# 저장 파일명들
model_file = out_dir / f"best_{model_name}_{timestamp}.pth"
metrics_file = out_dir / 'metrics.json'
meta_file = out_dir / 'metadata.json'
log_file = out_dir / 'train_log.txt'

# 실제로 모델 저장
torch.save(best_state, str(model_file))

# 또한 기존 OUT_PATH에도 복사(기존 워크플로우 유지)
try:
    shutil.copy2(str(model_file), str(OUT_PATH))
except Exception:
    # 복사 실패 시 무시
    pass

# 메트릭 및 메타데이터 저장
meta = {
    'project_root': str(PROJECT_ROOT),
    'train_dir': str(TRAIN_DIR),
    'out_dir': str(out_dir),
    'out_model_file': str(model_file),
    'out_path_copy': str(OUT_PATH),
    'hyperparams': { 'IMG_SIZE': IMG_SIZE, 'BATCH_SIZE': BATCH_SIZE, 'EPOCHS': EPOCHS, 'LR': LR, 'WEIGHT_DECAY': WEIGHT_DECAY, 'NUM_WORKERS': NUM_WORKERS, 'USE_PRETRAINED': USE_PRETRAINED },
    'best_epoch': int(best_state.get('epoch', 0)),
    'best_val_f1': float(best_state.get('val_f1', 0.0)),
    'timestamp': timestamp,
}
with open(metrics_file, 'w', encoding='utf-8') as f:
    json.dump(history, f, ensure_ascii=False, indent=2)
with open(meta_file, 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
with open(log_file, 'w', encoding='utf-8') as f:
    f.write('\n'.join(log_lines))

print('Saved best model and results →', out_dir)

# 새로 생성된 모델 경로를 검증 셀에서 사용할 수 있도록 전역 변수로 업데이트
# 이 셀을 재실행하면 이 변수가 설정됩니다.
LATEST_MODEL_PATH_FOR_VERIFICATION = str(model_file)
print('Verification path set to:', LATEST_MODEL_PATH_FOR_VERIFICATION)

Epoch 1/20: 100%|█| 370/370 [01:38<00:00,  3.76it/s, epoch=1 batch=370/370 loss=0.0085 lr=0.001000 m
Epoch 1/20: 100%|█| 370/370 [01:38<00:00,  3.76it/s, epoch=1 batch=370/370 loss=0.0085 lr=0.001000 m


Epoch [1/20] train_loss=0.1892 val_loss=0.1653 val_acc=0.9578 val_f1=0.0000 time=114.4s


Epoch 2/20: 100%|█| 370/370 [01:38<00:00,  3.75it/s, epoch=2 batch=370/370 loss=0.6758 lr=0.001000 m
Epoch 2/20: 100%|█| 370/370 [01:38<00:00,  3.75it/s, epoch=2 batch=370/370 loss=0.6758 lr=0.001000 m


Epoch [2/20] train_loss=0.1700 val_loss=0.1604 val_acc=0.9578 val_f1=0.0000 time=114.8s


Epoch 3/20: 100%|█| 370/370 [01:40<00:00,  3.69it/s, epoch=3 batch=370/370 loss=0.0271 lr=0.001000 m
Epoch 3/20: 100%|█| 370/370 [01:40<00:00,  3.69it/s, epoch=3 batch=370/370 loss=0.0271 lr=0.001000 m


...

Epoch [18/20] train_loss=0.1136 val_loss=0.1109 val_acc=0.9674 val_f1=0.0222 time=115.5s


Epoch 19/20: 100%|█| 370/370 [01:43<00:00,  3.57it/s, epoch=19 batch=370/370 loss=0.0170 lr=0.001000
Epoch 19/20: 100%|█| 370/370 [01:43<00:00,  3.57it/s, epoch=19 batch=370/370 loss=0.0170 lr=0.001000


Epoch [19/20] train_loss=0.1122 val_loss=0.1065 val_acc=0.9645 val_f1=0.0184 time=121.0s


Epoch 20/20: 100%|█| 370/370 [01:40<00:00,  3.69it/s, epoch=20 batch=370/370 loss=0.0124 lr=0.001000
Epoch 20/20: 100%|█| 370/370 [01:40<00:00,  3.69it/s, epoch=20 batch=370/370 loss=0.0124 lr=0.001000
                                                                                                    

Epoch [20/20] train_loss=0.1041 val_loss=0.1091 val_acc=0.9674 val_f1=0.0241 time=116.2s
Saved best model and results → d:\2ckvmfhwpr\skinseal-pythonAI\models\acne_model_20251029_110306
Verification path set to: d:\2ckvmfhwpr\skinseal-pythonAI\models\acne_model_20251029_110306\best_acne_model_20251029_110306.pth


In [22]:
# === 평가 셀: 베스트 체크포인트로 검증 수행 및 결과 저장 ===
# 사용법: 이 셀을 실행하면 val_loader 전체에 대해 예측을 수행하고 결과를 results 폴더에 저장합니다.
import torch, os, json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix

# results_dir는 이전 학습 셀에서 생성된 폴더여야 합니다. 없으면 OUT_PATH.parent를 기본으로 사용합니다.
results_dir = globals().get('results_dir', None) or str(OUT_PATH.parent)
results_dir = str(results_dir)
print('Using results_dir =', results_dir)

# 체크포인트 경로 탐색: LATEST_MODEL_PATH_FOR_VERIFICATION 우선, 그 다음 best_* 파일 검사
checkpoint_path = None
if 'LATEST_MODEL_PATH_FOR_VERIFICATION' in globals():
    p = Path(globals().get('LATEST_MODEL_PATH_FOR_VERIFICATION'))
    if p.exists():
        checkpoint_path = str(p)

if checkpoint_path is None:
    # best 파일 검색 (패턴: best_*.pth)
    cand = list(Path(results_dir).glob('best_*.pth'))
    if len(cand) > 0:
        # 최신 파일 선택
        cand.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        checkpoint_path = str(cand[0])

if checkpoint_path is None:
    raise FileNotFoundError(f'No checkpoint found in {results_dir}. Expected best_*.pth or LATEST_MODEL_PATH_FOR_VERIFICATION.')

print('Loading checkpoint:', checkpoint_path)
ck = torch.load(checkpoint_path, map_location=torch.device('cpu'))
# 체크포인트 구조에 따라 state_dict 키가 다를 수 있으니 유연하게 처리
if isinstance(ck, dict) and 'model_state_dict' in ck:
    state = ck['model_state_dict']
else:
    # assume ck itself is a state_dict
    state = ck

# 모델에 로드 (현재의 model 객체를 사용)
try:
    model.load_state_dict(state)
except Exception as e:
    # 일부 체크포인트는 키 접두사가 있을 수 있음(module.) — 자동 정리 시도
    new_state = {}
    for k, v in state.items():
        new_k = k.replace('module.', '') if k.startswith('module.') else k
        if new_k.startswith('model.'):
            new_k = new_k.replace('model.', '')
        new_state[new_k] = v
    model.load_state_dict(new_state)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

all_labels = []
all_preds = []
all_probs = []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        outs = model(imgs)
        probs = torch.softmax(outs, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)
        all_preds.extend(preds.tolist())
        all_probs.extend(probs.tolist())
        all_labels.extend(labels.numpy().tolist())

# metrics 및 저장
report = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True)
cm = confusion_matrix(all_labels, all_preds).tolist()
eval_out = { 'report': report, 'confusion_matrix': cm }

eval_path = os.path.join(results_dir, 'evaluation_results.json')
with open(eval_path, 'w', encoding='utf-8') as f:
    json.dump(eval_out, f, ensure_ascii=False, indent=2)

pd.DataFrame(all_probs, columns=class_names).to_csv(os.path.join(results_dir, 'predictions_probs.csv'), index=False)
pd.DataFrame({'label': all_labels, 'pred': all_preds}).to_csv(os.path.join(results_dir, 'predictions.csv'), index=False)

print('Saved evaluation to', eval_path)
print('Accuracy:', report.get('accuracy'))


Using results_dir = d:\2ckvmfhwpr\skinseal-pythonAI\models
Loading checkpoint: d:\2ckvmfhwpr\skinseal-pythonAI\models\acne_model_20251029_110306\best_acne_model_20251029_110306.pth


NameError: name 'class_names' is not defined

In [ ]:
# 저장된 가중치 간단 검증 (수정)
# 문제: 노트북 위치가 다르기 때문에 `acne_inference` 모듈을 못찾는 경우가 발생합니다.
# 해결: skinseal-pythonAI 경로를 sys.path에 추가한 뒤 임포트 시도. 실패하면 디렉터리 내용을 출력합니다.
import sys, os
sys.path.append(str(PROJECT_ROOT.parent / 'skinseal-pythonAI'))
print('appended to sys.path:', str(PROJECT_ROOT.parent / 'skinseal-pythonAI'))

try:
    from acne_inference import AcneInference
except Exception as e:
    print('Import failed:', repr(e))
    print('skinseal-pythonAI contents:', os.listdir(str(PROJECT_ROOT.parent / 'skinseal-pythonAI')))
    raise

# 검증할 모델 경로: 이전 셀에서 생성된 최신 모델 경로를 사용하거나, OUT_PATH를 기본값으로 사용
verification_path = OUT_PATH
if 'LATEST_MODEL_PATH_FOR_VERIFICATION' in locals() and Path(LATEST_MODEL_PATH_FOR_VERIFICATION).exists():
    verification_path = LATEST_MODEL_PATH_FOR_VERIFICATION
    
inf = AcneInference(weights_path=str(verification_path))
print('Loaded AcneInference with', verification_path)
# 현재 노트북에 설정된 EPOCHS 값 확인
print('EPOCHS configured in this notebook:', EPOCHS)
# 샘플 예측 (선택): 주석 해제 후 사용
# sample_img = PROJECT_ROOT / 'test' / 'Acne' / '0001.jpg'
# if sample_img.exists():
#     print(inf.predict(str(sample_img)))
# else:
#     print('샘플 이미지가 없습니다. 경로를 확인하세요.')

appended to sys.path: d:\2ckvmfhwpr\skinseal-pythonAI
Loaded AcneInference with d:\2ckvmfhwpr\skinseal-pythonAI\models\acne_model_20251029_100533\best_acne_model_20251029_100533.pth
EPOCHS configured in this notebook: 5
